# Notebook 06: Data Loading and Batching

## Overview

- **Duration**: ~2 hours
- **Prerequisites**: Notebooks 01-05
- **Learning Objectives**:
  1. Load and preprocess the TinyStories dataset
  2. Implement efficient tokenization and chunking
  3. Create PyTorch Dataset and DataLoader
  4. Understand sequence packing for language modeling

## Introduction

For language model training, we need to:
1. **Tokenize** the text into token IDs
2. **Chunk** into fixed-length sequences
3. **Create input/target pairs** (target is input shifted by 1)
4. **Batch** efficiently for GPU training

In [1]:
import sys
sys.path.insert(0, '../..')

import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


## Exercise 6.1: Load TinyStories Dataset (20 min)

In [2]:
from datasets import load_dataset

# Load TinyStories - a dataset of simple short stories
# Great for training small language models!
print("Loading TinyStories dataset...")

# For development, use a subset. For full training, remove the slice.
dataset = load_dataset("roneneldan/TinyStories", split="train[:50000]")

print(f"Loaded {len(dataset)} stories")
print(f"\nExample story:")
print(dataset[0]["text"][:500] + "...")

Loading TinyStories dataset...


Loaded 50000 stories

Example story:
One day, a little girl named Lily found a needle in her room. She knew it was difficult to play with it because it was sharp. Lily wanted to share the needle with her mom, so she could sew a button on her shirt.

Lily went to her mom and said, "Mom, I found this needle. Can you share it with me and sew my shirt?" Her mom smiled and said, "Yes, Lily, we can share the needle and fix your shirt."

Together, they shared the needle and sewed the button on Lily's shirt. It was not difficult for them b...


## Exercise 6.2: Tokenize the Dataset (30 min)

In [3]:
##the functions "encode" and "decode" we defined in notebook 01 making problems here, so we make a new tokenizer 
#Load or train a tokenizer
try:
    import pickle
    
    ## define a minimal dummy class to pickle the tokenizer
    #class BasicTokenizer:   # Must match the original class name
    #    pass
     
    tokenizer_path = Path("tokenizer.pkl")
    with open(tokenizer_path, "rb") as f:
        tokenizer = pickle.load(f)
    print(f"Loaded tokenizer with vocab size: {len(tokenizer.vocab)}")
except:
    print("Training new tokenizer...")
    from tokenizers import Tokenizer
    from tokenizers.models import BPE
    from tokenizers.trainers import BpeTrainer
    from tokenizers.pre_tokenizers import Whitespace

    tokenizer = Tokenizer(BPE())
    tokenizer.pre_tokenizer = Whitespace()
    
    trainer = BpeTrainer(vocab_size=10000)

    def batch_iterator():
        for text in dataset["text"][:10000]:
            yield text

    tokenizer.train_from_iterator(batch_iterator(), trainer)
    
    print(f"Trained tokenizer with vocab size: {tokenizer.get_vocab_size()}")

    """Save the tokenizer for future use"""
    import pickle
    save_path = Path("tokenizer.pkl")
    with open(save_path, "wb") as f:
        pickle.dump(tokenizer, f)

    print(f"Tokenizer saved to {save_path}")

Training new tokenizer...
Trained tokenizer with vocab size: 10000
Tokenizer saved to tokenizer.pkl


In [7]:
####solution 6.2
def tokenize_dataset(dataset, tokenizer, max_stories=None):
    """
    Tokenize all stories and concatenate into a single array.
    
    Args:
        dataset: HuggingFace dataset with 'text' column
        tokenizer: Tokenizer with encode() method
        max_stories: Optional limit on number of stories
        
    Returns:
        numpy array of token IDs
    """
    """Tokenize all stories and concatenate."""
    # TODO: Implement tokenization
    # O(N) where N is the number of stories (or max_stories)
    # 1. Iterate through stories (with tqdm for progress)
    tokenized_stories = []
    for story in tqdm(dataset["text"][:max_stories]):
        # 2. Encode each story
        tokens = tokenizer.encode(story).ids
        tokenized_stories.append(tokens) # Just appends pointer to list
    # 3. Concatenate all tokens
    all_tokens = np.concatenate([np.array(t) for t in tokenized_stories]) # single operation
    # 4. Return as numpy array
    return all_tokens

In [12]:
####test cell
# Tokenize the dataset
print("Tokenizing dataset...")
tokens = tokenize_dataset(dataset, tokenizer)

print(f"\nTotal tokens: {len(tokens):,}")
print(f"Unique tokens: {len(set(tokens)):,}")
print(f"First 50 tokens: {tokens[:50]}")

Tokenizing dataset...


100%|██████████| 50000/50000 [00:11<00:00, 4449.76it/s]



Total tokens: 10,489,966
Unique tokens: 9,348
First 50 tokens: [2.160e+02 1.560e+02 6.000e+00 4.800e+01 1.940e+02 2.350e+02 2.810e+02
 1.590e+02 3.600e+02 4.800e+01 3.266e+03 9.600e+01 1.210e+02 4.320e+02
 8.000e+00 1.250e+02 4.600e+02 1.000e+02 1.030e+02 2.535e+03 9.500e+01
 1.730e+02 1.500e+02 1.000e+02 5.560e+02 1.000e+02 1.030e+02 2.006e+03
 8.000e+00 1.590e+02 2.380e+02 9.500e+01 7.090e+02 9.200e+01 3.266e+03
 1.500e+02 1.210e+02 1.760e+02 6.000e+00 1.470e+02 1.400e+02 2.450e+02
 6.604e+03 4.800e+01 1.644e+03 1.060e+02 1.210e+02 1.957e+03 8.000e+00
 1.590e+02]


## Exercise 6.3: Create PyTorch Dataset (30 min)

In [13]:
####solution 6.3
class TextDataset(Dataset):
    """
    Dataset for language modeling.
    
    Returns (input, target) pairs where target is input shifted by 1.
    
    Args:
        tokens: Array of token IDs
        block_size: Sequence length (context window)
    """
    
    def __init__(self, tokens: np.ndarray, block_size: int):
        self.tokens = tokens
        self.block_size = block_size
    
    def __len__(self) -> int:
        # TODO: Return number of complete sequences
        return self.tokens.size - self.block_size 
        # the number of starting indices for block_size sequences
        # the last block_size tokens can't be a starting point for a full sequence, because
        # y should be x shifted by 1.
        # We need block_size + 1 tokens for each sample (input + 1 target token)
    
    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        """
        Get a single training example.
        
        Returns:
            x: Input tokens of shape (block_size,)
            y: Target tokens of shape (block_size,) - shifted by 1
        """
        # TODO: Implement __getitem__
        # 1. Get block_size + 1 tokens starting at idx
        tokens = self.tokens[idx : idx + self.block_size + 1]
        # 2. x = first block_size tokens
        x = tokens[:-1]  # first block_size tokens
        # 3. y = last block_size tokens (shifted by 1)
        y = tokens[1:]   # last block_size tokens (shifted by 1)
        # 4. Convert to torch tensors
        x = torch.tensor(x, dtype=torch.long)
        y = torch.tensor(y, dtype=torch.long)
            
        return x, y

In [14]:
####test cell
# Test the dataset
block_size = 256
dataset_train = TextDataset(tokens, block_size)

print(f"Dataset size: {len(dataset_train):,} samples")
print(f"Block size: {block_size}")

# Get a sample
x, y = dataset_train[0]
print(f"\nSample shapes: x={x.shape}, y={y.shape}")
print(f"x[:10]: {x[:10].tolist()}")
print(f"y[:10]: {y[:10].tolist()}")

# Verify shift
assert torch.equal(x[1:], y[:-1]), "y should be x shifted by 1"
print("\n✓ Shift verified: y[i] = x[i+1]")

Dataset size: 10,489,710 samples
Block size: 256

Sample shapes: x=torch.Size([256]), y=torch.Size([256])
x[:10]: [216, 156, 6, 48, 194, 235, 281, 159, 360, 48]
y[:10]: [156, 6, 48, 194, 235, 281, 159, 360, 48, 3266]

✓ Shift verified: y[i] = x[i+1]


## Exercise 6.4: Train/Validation Split (15 min)

In [15]:
####solution 6.4
def create_train_val_split(tokens: np.ndarray, val_ratio: float = 0.1):
    """
    Split tokens into train and validation sets.
    
    Args:
        tokens: Array of token IDs
        val_ratio: Fraction of data for validation
        
    Returns:
        train_tokens, val_tokens
    """
    # TODO: Split the tokens
    # Simple approach: last val_ratio of tokens for validation
    split_idx = int(len(tokens) * (1 - val_ratio))
    train_tokens = tokens[:split_idx]
    val_tokens = tokens[split_idx:]
    return train_tokens, val_tokens

In [16]:
####test cell
# Create split
train_tokens, val_tokens = create_train_val_split(tokens, val_ratio=0.1)

print(f"Train tokens: {len(train_tokens):,}")
print(f"Val tokens: {len(val_tokens):,}")

# Create datasets
train_dataset = TextDataset(train_tokens, block_size)
val_dataset = TextDataset(val_tokens, block_size)

print(f"\nTrain samples: {len(train_dataset):,}")
print(f"Val samples: {len(val_dataset):,}")

Train tokens: 9,440,969
Val tokens: 1,048,997

Train samples: 9,440,713
Val samples: 1,048,741


## Exercise 6.5: Create DataLoaders (20 min)

In [18]:
####solution 6.5
def create_dataloaders(
    train_dataset: Dataset,
    val_dataset: Dataset,
    batch_size: int = 32,
    num_workers: int = 0,
):
    """
    Create DataLoaders for training and validation.
    
    Args:
        train_dataset: Training dataset
        val_dataset: Validation dataset
        batch_size: Batch size
        num_workers: Number of worker processes
        
    Returns:
        train_loader, val_loader
    """
    # TODO: Create DataLoaders
    # - Train: shuffle=True
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True)
    # - Val: shuffle=False
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, drop_last=True)
    # - Both: drop_last=True (for consistent batch sizes)
    return train_loader, val_loader

In [19]:
####test cell
# Create dataloaders
batch_size = 32
train_loader, val_loader = create_dataloaders(train_dataset, val_dataset, batch_size)

print(f"Train batches: {len(train_loader):,}")
print(f"Val batches: {len(val_loader):,}")

# Test a batch
x_batch, y_batch = next(iter(train_loader))
print(f"\nBatch shapes: x={x_batch.shape}, y={y_batch.shape}")

Train batches: 295,022
Val batches: 32,773

Batch shapes: x=torch.Size([32, 256]), y=torch.Size([32, 256])


## Visualize the Data

In [20]:
# Decode a sample to see what the model will learn
x, y = train_dataset[0]

input_text = tokenizer.decode(x.tolist())
target_text = tokenizer.decode(y.tolist())

print("Input text:")
print(input_text[:500])
print("\n" + "="*50)
print("\nTarget text (should be shifted by 1 token):")
print(target_text[:500])

Input text:
One day , a little girl named Lily found a needle in her room . She knew it was difficult to play with it because it was sharp . Lily wanted to share the needle with her mom , so she could sew a button on her shirt . Lily went to her mom and said , " Mom , I found this needle . Can you share it with me and sew my shirt ?" Her mom smiled and said , " Yes , Lily , we can share the needle and fix your shirt ." Together , they shared the needle and sewed the button on Lily ' s shirt . It was not dif


Target text (should be shifted by 1 token):
day , a little girl named Lily found a needle in her room . She knew it was difficult to play with it because it was sharp . Lily wanted to share the needle with her mom , so she could sew a button on her shirt . Lily went to her mom and said , " Mom , I found this needle . Can you share it with me and sew my shirt ?" Her mom smiled and said , " Yes , Lily , we can share the needle and fix your shirt ." Together , they shared the needle 

## Save Preprocessed Data (Optional)

In [21]:
# Save for faster loading later
save_path = Path("data")
save_path.mkdir(exist_ok=True)

np.save(save_path / "train_tokens.npy", train_tokens)
np.save(save_path / "val_tokens.npy", val_tokens)

print(f"Saved preprocessed data to {save_path}/")

Saved preprocessed data to data/


## Summary

### What You Learned

1. **TinyStories**: A dataset perfect for training small LMs
2. **Tokenization**: Convert text to token IDs
3. **Chunking**: Split into fixed-length sequences
4. **Input/Target pairs**: Target is input shifted by 1
5. **DataLoader**: Efficient batching for GPU training

### Next: Training Loop

In Notebook 07, we'll implement the training loop and train our GPT model!